# Tree-of-Thought (ToT) Reasoning Demo

This notebook tests the Tree-of-Thought reasoning strategy added to `MedicalLLMWrapper`.

**How ToT works in this wrapper:**
1. **Branch** — sample `n_branches` independent reasoning thoughts from the prompt
2. **Score** — rank each thought by mean log-probability (model coherence proxy, no extra generation calls)
3. **Select** — keep the highest-scoring thought
4. **Solve** — run constrained answer generation conditioned on the best thought
5. **Explain** — generate final rationale

**Output format:**
```
Answer: A
Thought: <best reasoning branch selected from N candidates>
Rationale: <final explanation>
```

**Sections:**
1. Setup and Installation
2. Basic ToT — Apollo-2B MCQ
3. ToT Branch Inspection — view all branches and scores
4. CoT vs ToT Comparison — same prompt, both strategies
5. ToT with MedGemma — larger model
6. ToT on Yes/No Questions
7. Effect of `n_branches` on answer quality
8. Batch processing with ToT
9. Summary

## 1. Setup and Installation

In [ ]:
!pip install torch transformers accelerate hf_xet -q
print("✓ Packages installed successfully!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import os

project_path = "/content/drive/MyDrive/DATA 298A/sjsu-data298-main"

if project_path not in sys.path:
    sys.path.append(project_path)

print(f"✓ Project path: {project_path}")
print(f"✓ Path exists: {os.path.exists(project_path)}")

if os.path.exists(project_path):
    contents = os.listdir(project_path)
    print(f"✓ Contents: {contents}")
    if "medical_llm_wrapper.py" in contents:
        print("✓ medical_llm_wrapper.py found!")
    else:
        print("⚠️  WARNING: medical_llm_wrapper.py NOT FOUND!")
else:
    print(f"⚠️  ERROR: Project path does not exist!")

In [ ]:
import warnings
warnings.filterwarnings('once')

project_path = "/content/drive/MyDrive/DATA 298A/sjsu-data298-main"
if project_path not in sys.path:
    sys.path.insert(0, project_path)

try:
    from medical_llm_wrapper import MedicalLLMWrapper, load_medical_llm
    import torch
    print("✓ Medical LLM Wrapper imported successfully!")
    print(f"✓ CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
        print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except ModuleNotFoundError as e:
    print(f"❌ Error: {e}")
    print("\n📋 Steps to fix:")
    print("1. Runtime → Restart runtime")
    print("2. Re-run all cells from the beginning")
    raise

## 2. Basic ToT — Apollo-2B MCQ

Load Apollo-2B (fp16, no auth token needed) and run a medical MCQ using Tree-of-Thought.
- Enable ToT with `set_reasoning_strategy('tot')`
- Default config: 3 branches, 80 tokens per branch

In [ ]:
print("=" * 80)
print("TEST 1: Apollo-2B — MCQ with Tree-of-Thought")
print("=" * 80)

apollo = load_medical_llm(
    "FreedomIntelligence/Apollo-2B",
    device="cuda",
    torch_dtype=torch.float16
)

apollo.set_task("mcq")
apollo.set_mode("answer_rationale")
apollo.set_reasoning_strategy("tot")

print("\n[Model Information]")
info = apollo.get_model_info()
for key, value in info.items():
    if key != "num_parameters":
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value:,}")

mcq_prompt = """A 55-year-old patient presents with persistent cough, hemoptysis, and unintentional weight loss.
Chest X-ray shows a mass in the right upper lobe. What is the most likely diagnosis?

A) Tuberculosis
B) Lung cancer
C) Pneumonia
D) Pulmonary embolism

Answer:"""

print("\n" + "=" * 80)
print("[Running Tree-of-Thought generation...]")
print("=" * 80)

response = apollo.generate(mcq_prompt)

print("\n" + "=" * 80)
print("[FINAL OUTPUT]")
print("=" * 80)
print(response)
print("\n✓ Test 1 complete!")

## 3. ToT Branch Inspection

After a ToT generation, the wrapper stores all branches and their log-prob scores in:
- `wrapper.last_thoughts` — list of all generated reasoning branches
- `wrapper.last_thought_scores` — log-prob score for each branch (higher = more coherent)
- `wrapper.last_best_thought` — the branch that was selected

This cell prints a full breakdown of the decision.

In [ ]:
print("=" * 80)
print("BRANCH INSPECTION — All ToT Candidates")
print("=" * 80)

thoughts = apollo.last_thoughts
scores = apollo.last_thought_scores
best_thought = apollo.last_best_thought

if thoughts is None:
    print("⚠️  No ToT data found. Run a ToT generation first.")
else:
    best_idx = scores.index(max(scores))

    for i, (thought, score) in enumerate(zip(thoughts, scores)):
        marker = "★ SELECTED" if i == best_idx else f"  branch {i+1}"
        print(f"\n[{marker}] Score: {score:.4f}")
        print("-" * 60)
        print(thought)

    print("\n" + "=" * 80)
    score_range = max(scores) - min(scores)
    print(f"Score range: {min(scores):.4f} → {max(scores):.4f} (spread: {score_range:.4f})")
    print(f"Best branch index: {best_idx + 1} of {len(thoughts)}")
    print("=" * 80)

## 4. CoT vs ToT Comparison

Run the **same prompt** with both Chain-of-Thought and Tree-of-Thought on Apollo-2B
and compare:
- The reasoning process
- The final answer
- The rationale

In [ ]:
print("=" * 80)
print("TEST 2: CoT vs ToT — Side-by-Side Comparison")
print("=" * 80)

comparison_prompt = """A 45-year-old man with type 2 diabetes presents with a foot ulcer that has not healed
in 3 weeks. His HbA1c is 10.2%. Which of the following is the most important initial step?

A) Prescribe oral antibiotics
B) Optimize glycemic control
C) Schedule amputation
D) Apply topical corticosteroids

Answer:"""

print(f"\nPrompt:\n{comparison_prompt}")

# --- Chain-of-Thought ---
print("\n" + "=" * 80)
print("[Strategy: Chain-of-Thought (CoT)]")
print("=" * 80)
apollo.set_reasoning_strategy("cot")
cot_response = apollo.generate(comparison_prompt)
print(cot_response)

# --- Tree-of-Thought ---
print("\n" + "=" * 80)
print("[Strategy: Tree-of-Thought (ToT)]")
print("=" * 80)
apollo.set_reasoning_strategy("tot")
tot_response = apollo.generate(comparison_prompt)
print(tot_response)

# --- Side-by-side summary ---
print("\n" + "=" * 80)
print("[SUMMARY]")
print("=" * 80)
cot_answer = cot_response.split('\n')[0].replace('Answer: ', '')
tot_answer = tot_response.split('\n')[0].replace('Answer: ', '')
print(f"CoT answer: {cot_answer}")
print(f"ToT answer: {tot_answer}")
print(f"Answers agree: {cot_answer == tot_answer}")
print("✓ Test 2 complete!")

## 5. ToT with MedGemma

Test ToT on MedGemma-4B-IT, the larger and more capable medical model.
- Requires a Hugging Face auth token (gated model)
- Automatically uses float32
- Expect slower generation due to model size

In [ ]:
# Set your Hugging Face token here if needed
HF_TOKEN = None  # e.g. "hf_xxxx..."

print("=" * 80)
print("TEST 3: MedGemma-4B-IT — MCQ with Tree-of-Thought")
print("=" * 80)

# Clean up Apollo first to free VRAM
del apollo
torch.cuda.empty_cache()

medgemma = load_medical_llm(
    "google/medgemma-4b-it",
    device="cuda",
    token=HF_TOKEN
    # torch_dtype omitted — wrapper auto-uses float32 for MedGemma
)

medgemma.set_task("mcq")
medgemma.set_mode("answer_rationale")
medgemma.set_reasoning_strategy("tot")
medgemma.configure_tot(n_branches=3, max_thought_tokens=80)

print("\n[Model Information]")
info = medgemma.get_model_info()
for key, value in info.items():
    if key != "num_parameters":
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value:,}")

medgemma_prompt = """A 28-year-old woman is 32 weeks pregnant and presents with sudden severe headache,
visual disturbances, and blood pressure of 160/110 mmHg. What is the most appropriate management?

A) Immediate cesarean section
B) Administer magnesium sulfate and antihypertensives
C) Prescribe bed rest and follow up in one week
D) Perform lumbar puncture to rule out meningitis

Answer:"""

print("\n" + "=" * 80)
print("[Running Tree-of-Thought generation...]")
print("=" * 80)

response = medgemma.generate(medgemma_prompt)

print("\n" + "=" * 80)
print("[FINAL OUTPUT]")
print("=" * 80)
print(response)
print("\n✓ Test 3 complete!")

In [ ]:
# Full branch breakdown for MedGemma
print("=" * 80)
print("BRANCH INSPECTION — MedGemma ToT Candidates")
print("=" * 80)

thoughts = medgemma.last_thoughts
scores = medgemma.last_thought_scores

if thoughts:
    best_idx = scores.index(max(scores))
    for i, (thought, score) in enumerate(zip(thoughts, scores)):
        marker = "★ SELECTED" if i == best_idx else f"  branch {i+1}"
        print(f"\n[{marker}] Score: {score:.4f}")
        print("-" * 60)
        print(thought)

    print("\n" + "=" * 80)
    print(f"Score spread: {max(scores) - min(scores):.4f}")
    print("=" * 80)

## 6. ToT on Yes/No Questions

Tree-of-Thought also works for binary Yes/No tasks (`task_type='yn'`).

In [ ]:
print("=" * 80)
print("TEST 4: MedGemma — Yes/No Question with Tree-of-Thought")
print("=" * 80)

medgemma.set_task("yn")
medgemma.set_reasoning_strategy("tot")

yn_prompt = """ACE inhibitors are contraindicated in patients with bilateral renal artery stenosis.

A) Yes
B) No

Answer:"""

print(f"\nPrompt:\n{yn_prompt}")
print("\n[Running ToT...]")

response = medgemma.generate(yn_prompt)

print("\n" + "=" * 80)
print("[FINAL OUTPUT]")
print("=" * 80)
print(response)
print("\n✓ Test 4 complete!")

## 7. Effect of `n_branches` on Reasoning

Compare how increasing the number of branches affects the selected reasoning path.
- More branches = broader exploration of the reasoning space
- Trade-off: each branch requires one additional generation call

This runs the same prompt with 2, 3, and 5 branches.

In [ ]:
print("=" * 80)
print("TEST 5: Effect of n_branches on ToT Reasoning")
print("=" * 80)

medgemma.set_task("mcq")
medgemma.set_reasoning_strategy("tot")

branch_prompt = """A patient on long-term heparin therapy develops thrombocytopenia.
Which condition should be suspected?

A) Disseminated intravascular coagulation
B) Heparin-induced thrombocytopenia
C) Immune thrombocytopenic purpura
D) Thrombotic thrombocytopenic purpura

Answer:"""

branch_counts = [2, 3, 5]
branch_results = {}

for n in branch_counts:
    print(f"\n{'=' * 60}")
    print(f"n_branches = {n}")
    print("=" * 60)
    medgemma.configure_tot(n_branches=n, max_thought_tokens=80)
    result = medgemma.generate(branch_prompt)
    branch_results[n] = {
        "response": result,
        "answer": medgemma.last_answer,
        "best_thought": medgemma.last_best_thought,
        "scores": medgemma.last_thought_scores,
    }
    print(result)
    print(f"\nScore spread: {max(medgemma.last_thought_scores) - min(medgemma.last_thought_scores):.4f}")

# Summary table
print("\n" + "=" * 80)
print("[SUMMARY: n_branches comparison]")
print(f"{'n_branches':<12} {'Answer':<10} {'Best score':<14} {'Score spread'}")
print("-" * 50)
for n, data in branch_results.items():
    best_score = max(data['scores'])
    spread = max(data['scores']) - min(data['scores'])
    print(f"{n:<12} {data['answer']:<10} {best_score:<14.4f} {spread:.4f}")
print("=" * 80)
print("✓ Test 5 complete!")

## 8. Batch Processing with ToT

ToT works transparently with `batch_generate()` — each prompt goes through the full
branch/score/select pipeline. Note: batch time scales with `n_branches`.

In [ ]:
print("=" * 80)
print("TEST 6: Batch Processing with ToT")
print("=" * 80)

medgemma.set_task("mcq")
medgemma.set_reasoning_strategy("tot")
medgemma.configure_tot(n_branches=3, max_thought_tokens=60)  # shorter thoughts for speed

batch_prompts = [
    """Which vitamin deficiency causes scurvy?
A) Vitamin A
B) Vitamin B12
C) Vitamin C
D) Vitamin D

Answer:""",

    """What is the first-line treatment for Helicobacter pylori infection?
A) Omeprazole alone
B) Triple therapy (PPI + amoxicillin + clarithromycin)
C) Metronidazole alone
D) Bismuth subsalicylate alone

Answer:""",

    """Which enzyme is deficient in phenylketonuria (PKU)?
A) Tyrosine hydroxylase
B) Phenylalanine hydroxylase
C) Homogentisate oxidase
D) Dopamine beta-hydroxylase

Answer:"""
]

print(f"\nProcessing {len(batch_prompts)} prompts with ToT (n_branches=3)...")
print("=" * 80)

results = medgemma.batch_generate(batch_prompts, show_progress=True)

print("\n" + "=" * 80)
print("[BATCH RESULTS]")
print("=" * 80)

questions = [
    "Vitamin deficiency → scurvy?",
    "First-line treatment for H. pylori?",
    "Enzyme deficient in PKU?"
]
correct_answers = ["C", "B", "B"]

for i, (question, result, correct) in enumerate(zip(questions, results, correct_answers), 1):
    answer_line = result.split('\n')[0]
    predicted = answer_line.replace('Answer: ', '').strip()
    status = "✓" if predicted == correct else "✗"
    print(f"\n{i}. {question}")
    print(f"   Predicted: {predicted}  |  Correct: {correct}  {status}")
    # Print the thought used
    thought_line = [l for l in result.split('\n') if l.startswith('Thought:')]
    if thought_line:
        thought_text = thought_line[0].replace('Thought: ', '')[:100]
        print(f"   Thought:   {thought_text}...")

print("\n" + "=" * 80)
print("✓ Test 6 complete!")

## 9. Summary

### What was tested:

| Test | Model | Task | Branches | Purpose |
|------|-------|------|----------|---------|
| 1 | Apollo-2B | MCQ | 3 | Basic ToT sanity check |
| 2 | Apollo-2B | MCQ | 3 | CoT vs ToT side-by-side |
| 3 | MedGemma-4B | MCQ | 3 | ToT on larger model |
| 4 | MedGemma-4B | Y/N | 3 | ToT on binary task |
| 5 | MedGemma-4B | MCQ | 2/3/5 | Effect of n_branches |
| 6 | MedGemma-4B | MCQ | 3 | Batch processing |

### Key API:

```python
wrapper.set_reasoning_strategy('tot')          # enable Tree-of-Thought
wrapper.configure_tot(n_branches=3,            # optional tuning
                      max_thought_tokens=80)
response = wrapper.generate(prompt)

# Post-generation inspection
wrapper.last_thoughts        # all candidate branches (list[str])
wrapper.last_thought_scores  # log-prob score per branch (list[float])
wrapper.last_best_thought    # selected branch (str)
```

### Output format:
```
Answer: B
Thought: <best reasoning branch chosen from N candidates>
Rationale: <final explanation conditioned on best thought>
```

In [ ]:
# Final cleanup
import gc

try:
    del medgemma
except NameError:
    pass
try:
    del apollo
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print("✓ All tests complete!")
print("✓ Memory cleaned up")
print("\n" + "=" * 80)
print("Tree-of-Thought Demo — SUCCESS!")
print("=" * 80)